## 1. Importar librerías

In [56]:
import pandas as pd
import numpy as np
import joblib
import os

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

## 2. Cargar datasets

In [57]:
test = pd.read_csv("../data/processed/test.csv")

TARGET = "helada_24h"

X_test = test.drop(columns=["fecha_hora", TARGET])
y_test = test[TARGET]

## 3. Cargar modelos entrenados

In [58]:
rf = joblib.load("../models/random_forest.pkl")
svm = joblib.load("../models/svm.pkl")
mlp = joblib.load("../models/neural_network.pkl")

## 4. Probabilidades (Clave del ensamble)

In [59]:
rf_proba = rf.predict_proba(X_test)[:, 1]
svm_proba = svm.predict_proba(X_test)[:, 1]
mlp_proba = mlp.predict_proba(X_test)[:, 1]

## 5. Ensamble Híbrido (Soft voting ponderado)

In [60]:
weights = {
    "rf": 0.35,
    "svm": 0.40,
    "mlp": 0.25
}

y_proba_ensemble = (
    weights["rf"] * rf_proba +
    weights["svm"] * svm_proba +
    weights["mlp"] * mlp_proba
)

## 6. Probar diferentes thresholds

In [61]:
thresholds = [0.30, 0.35, 0.40, 0.45, 0.50]

results = []

for t in thresholds:
    y_pred = (y_proba_ensemble >= t).astype(int)

    results.append({
        "threshold": t,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred)
    })

results_df = pd.DataFrame(results)
results_df

,threshold,accuracy,precision,recall,f1
0,0.30,0.800339,0.607486,0.778598,0.682480
1,0.35,0.805763,0.616959,0.778598,0.688418
2,0.40,0.811186,0.628514,0.769988,0.692095
3,0.45,0.818305,0.643226,0.765068,0.698876
4,0.50,0.823729,0.655685,0.758918,0.703535


## 7. Elegir mejor threshold

In [62]:
best_row = results_df.sort_values("f1", ascending=False).iloc[0]
best_threshold = best_row["threshold"]

best_row

threshold    0.500000
accuracy     0.823729
precision    0.655685
recall       0.758918
f1           0.703535
Name: 4, dtype: float64

## 8. Matriz de confusión final

In [63]:
y_final = (y_proba_ensemble >= best_threshold).astype(int)

confusion_matrix(y_test, y_final)

array([[1813,  324],
       [ 196,  617]])

## 9. Guardar predicciones

In [64]:
os.makedirs("../results", exist_ok=True)

pred_df = pd.DataFrame({
    "y_true": y_test,
    "y_pred": y_final,
    "proba": y_proba_ensemble
})

pred_df.to_csv("../results/predicciones_ensemble.csv", index=False)

## 10. Guardar métricas del ensamble

In [65]:
ensemble_metrics = pd.DataFrame(results)
ensemble_metrics.to_csv("../results/metricas_ensemble.csv", index=False)

ensemble_metrics

,threshold,accuracy,precision,recall,f1
0,0.30,0.800339,0.607486,0.778598,0.682480
1,0.35,0.805763,0.616959,0.778598,0.688418
2,0.40,0.811186,0.628514,0.769988,0.692095
3,0.45,0.818305,0.643226,0.765068,0.698876
4,0.50,0.823729,0.655685,0.758918,0.703535


## 11. Guardar modelo del ensamble

In [66]:
ensemble_model = {
    "rf": rf,
    "svm": svm,
    "mlp": mlp,
    "weights": weights,
    "threshold": best_threshold
}

joblib.dump(ensemble_model, "../models/ensemble.pkl")

['../models/ensemble.pkl']